# Lọc lấy các thông tin quan trọng

In [ ]:
from pathlib import Path
import json

INPUT_DIR = Path("data/stage1/subjects")
OUTPUT_DIR = Path("data/stage2")

METADATA_FIELDS = [
    "syllabus_name",
    "course_name_english",
    "subject_code",
    "description",
    "tools",
]
CLO_FIELDS = ["clo_name", "clo_details"]
SCHEDULE_FIELDS = ["topic", "lo"]
ASSESSMENT_FIELDS = ["category", "part", "weight", "clo"]


def select_fields(record, fields):
    return {field: record.get(field) for field in fields}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_files = 0

for input_path in sorted(INPUT_DIR.glob("*.json")):
    with input_path.open("r", encoding="utf-8") as file:
        source = json.load(file)

    content = source.get("content") or {}
    metadata = content.get("metadata") or {}
    clos = content.get("clos") or []
    schedule = content.get("schedule") or []
    assessments = content.get("assessments") or []

    result = {
        "metadata": select_fields(metadata, METADATA_FIELDS),
        "clos": [select_fields(item, CLO_FIELDS) for item in clos],
        "schedule": [select_fields(item, SCHEDULE_FIELDS) for item in schedule],
        "assessments": [
            select_fields(item, ASSESSMENT_FIELDS) for item in assessments
        ],
    }

    output_path = OUTPUT_DIR / input_path.name
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(result, file, ensure_ascii=False, indent=2)
        file.write("\n")

    processed_files += 1

print(f"Created {processed_files} files in {OUTPUT_DIR}")


# Slipt metadata, clos, map topic được học với CLOs

In [7]:
from collections import defaultdict
from pathlib import Path
import json
import re

INPUT_DIR = Path("data/stage2")
OUTPUT_DIR = Path("data/stage3")


def extract_clo_names(value, available_clos=None):
    text = str(value or "").strip().replace("–", "-").replace("—", "-")
    if not text:
        return []

    available_clos = available_clos or []
    if re.search(r"all\s+(?:clo|lo)s?", text, flags=re.IGNORECASE):
        return list(available_clos)

    names = []
    range_pattern = re.compile(
        r"(?:CLO|LO)\s*(\d+)\s*-\s*(?:CLO|LO)?\s*(\d+)",
        flags=re.IGNORECASE,
    )
    ranged_text = text
    for match in range_pattern.finditer(text):
        start, end = int(match.group(1)), int(match.group(2))
        step = 1 if start <= end else -1
        names.extend(f"CLO{number}" for number in range(start, end + step, step))
        ranged_text = ranged_text.replace(match.group(0), " ")

    for number in re.findall(r"(?:CLO|LO)\s*(\d+)", ranged_text, flags=re.IGNORECASE):
        name = f"CLO{number}"
        if name not in names:
            names.append(name)

    return names


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_subjects = 0

for input_path in sorted(INPUT_DIR.glob("*.json")):
    subject_code = input_path.stem
    subject_dir = OUTPUT_DIR / subject_code
    subject_dir.mkdir(parents=True, exist_ok=True)

    with input_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    for field in ("metadata", "clos", "assessments"):
        output_path = subject_dir / f"{field}.json"
        with output_path.open("w", encoding="utf-8") as file:
            json.dump(data.get(field, {} if field == "metadata" else []), file, ensure_ascii=False, indent=2)
            file.write("\n")

    clo_names = []
    for item in data.get("clos", []):
        for clo_name in extract_clo_names(item.get("clo_name")):
            if clo_name not in clo_names:
                clo_names.append(clo_name)

    topics_by_clo = defaultdict(list)
    for schedule_item in data.get("schedule", []):
        topic = schedule_item.get("topic")
        if not topic:
            continue

        schedule_clo_names = extract_clo_names(
            schedule_item.get("lo"), available_clos=clo_names
        )
        for clo_name in schedule_clo_names:
            if topic not in topics_by_clo[clo_name]:
                topics_by_clo[clo_name].append(topic)

    all_clo_names = list(dict.fromkeys(clo_names + list(topics_by_clo)))
    topics_by_clo_json = {
        clo_name: topics_by_clo.get(clo_name, []) for clo_name in all_clo_names
    }

    output_path = subject_dir / "topics_by_clo.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(topics_by_clo_json, file, ensure_ascii=False, indent=2)
        file.write("\n")

    processed_subjects += 1

print(f"Created stage3 data for {processed_subjects} subjects in {OUTPUT_DIR}")


Created stage3 data for 49 subjects in data/stage3
